[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MinhEiden/ML-Assignment-/blob/main/notebooks/01_eda_exploration.ipynb)

# BÀI TẬP LỚN HỌC MÁY - BÁO CÁO EDA PHÂN TÍCH THEO NHÓM ĐẶC TRƯNG
## Tập dữ liệu: HR Analytics: Job Change of Data Scientists (`aug_train.csv`)
---
### BẢNG THỐNG NHẤT PHÂN NHÓM ĐẶC TRƯNG & LOẠI BIỂU ĐỒ TRỰC QUAN HÓA:
| Nhóm | Tên đặc trưng | Kiểu dữ liệu | Biểu đồ phù hợp nhất | Mục đích phân tích & Đóng góp cho Pipeline |
| :--- | :--- | :--- | :--- | :--- |
| **Nhóm 1: Biến mục tiêu** | `target` | Binary (0, 1) | **Bar Chart + Donut/Pie Chart** | Đánh giá mức độ mất cân bằng lớp (Class Imbalance: ~24.9% Target=1). Định hướng chiến lược resample hoặc điều chỉnh class weight. |
| **Nhóm 2: Biến số / Liên tục** | `city_development_index`, `training_hours` | Float / Int | **Histogram (KDE) + Boxplot theo Target** | Khảo sát phân phối lệch (skewness), ngoại lai (outliers) và khả năng phân tách nhãn mục tiêu. |
| **Nhóm 3: Danh nghĩa ít giá trị** | `gender`, `company_type`, `major_discipline`, `relevent_experience`, `enrolled_university` | Categorical Nominal | **Phương án A: Bar Chart (Count & %) + Bar Chart (Target=1 %)** | Xem quy mô từng danh mục kèm tỷ lệ muốn đổi việc để đo lường tính đại diện và mức độ ảnh hưởng đến Target. |
| **Nhóm 4: Thứ bậc (Ordinal)** | `education_level`, `company_size`, `experience`, `last_new_job` | Categorical Ordinal | **Ordered Bar Chart + Tỷ lệ Target=1 (Line Chart cho `experience`)** | Đánh giá xu hướng đơn điệu (monotonic trend). Riêng `experience` (23 mốc) dùng Line Chart để nhìn rõ độ dốc giảm dần. |
| **Nhóm 5: Danh nghĩa nhiều giá trị** | `city` (123 thành phố) | High-cardinality Nominal | **Top 15 Horizontal Bar Chart + Kiểm chứng 1-1 với CDI** | Đánh giá các đô thị tập trung ứng viên. Chứng minh quan hệ 1-1 với `city_development_index` để loại bỏ `city` tránh bùng nổ số chiều. |
| **Nhóm 6: Dữ liệu khuyết & Tương quan** | Tất cả các cột có missing values | Khuyết & Tương quan | **Missing Bar + MNAR Analysis + Spearman Heatmap (7x7)** | Chứng minh cơ chế khuyết MNAR ở `company_*` để không drop dữ liệu; tính tương quan thứ bậc với Target. |
| **Nhóm 7: Khảo sát không gian PCA** | Toàn bộ đặc trưng đầu vào | Đa chiều (62 features sau One-Hot) | **PCA Scree Plot (Cumulative Variance Ratio)** | Bám sát mục 4.1 Đề bài BTL: Xác định số thành phần chính k để giữ 90% và 95% lượng thông tin (phương sai). |


In [ ]:
# ==============================================================================
# 0. CẤU HÌNH MÔI TRƯỜNG TỰ ĐỘNG (HỖ TRỢ CẢ LOCAL VÀ GOOGLE COLAB)
# ==============================================================================
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Tự động nhận diện và cấu hình khi chạy trên Google Colab
if "google.colab" in sys.modules:
    REPO_NAME = "ML-Assignment-"
    REPO_URL = "https://github.com/MinhEiden/ML-Assignment-.git"
    if not os.path.exists(REPO_NAME):
        os.system(f"git clone {REPO_URL}")
    repo_path = os.path.abspath(REPO_NAME)
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)
    # Chuyển thư mục làm việc vào repo/notebooks để đường dẫn tương đối hoạt động đồng nhất
    notebooks_dir = os.path.join(repo_path, "notebooks")
    if os.path.exists(notebooks_dir):
        os.chdir(notebooks_dir)
    print(f"-> Đã cấu hình Google Colab thành công. Thư mục làm việc: {os.getcwd()}")
else:
    # Môi trường Local: thêm thư mục gốc dự án vào sys.path
    project_root = os.path.abspath("..") if os.path.exists("../modules") else os.path.abspath(".")
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

from modules.eda_viz import (
    get_missing_stats,
    plot_target_distribution,
    plot_numerical_feature,
    plot_nominal_feature,
    plot_ordinal_feature,
    plot_high_cardinality_feature,
    plot_missing_values,
    plot_missingness_vs_target,
    plot_correlation_matrix,
    check_city_cdi_relationship,
    plot_pca_scree_plot
)

# Nạp dữ liệu từ thư mục data/
DATA_PATH = "../data/aug_train.csv" if os.path.exists("../data/aug_train.csv") else "data/aug_train.csv"
df = pd.read_csv(DATA_PATH)
print(f"Nạp thành công tập dữ liệu: {df.shape[0]:,} dòng và {df.shape[1]} cột")
df.head()


---
## NHÓM 1: BIẾN MỤC TIÊU (`target`)
**Biểu đồ sử dụng:** Bar Chart (Số lượng & %) kết hợp Donut/Pie Chart (Tỷ trọng cơ cấu).
- Đánh giá sự mất cân bằng giữa nhãn 0 (Không đổi việc) và nhãn 1 (Muốn đổi việc).


In [ ]:
plot_target_distribution(df, target_col='target', save_path='../reports/figures/group1_target_distribution.png', show=True)


---
## NHÓM 2: BIẾN SỐ / LIÊN TỤC (NUMERICAL FEATURES)
**Biểu đồ sử dụng:** Histogram + KDE (mật độ phân phối) và Boxplot (phát hiện ngoại lai, so sánh trung vị theo target).
- Histogram sử dụng `multiple='dodge'` xếp các cột cạnh nhau rõ ràng, phân biệt màu sắc mạch lạc giữa nhãn 0 và 1.
- Boxplot làm nổi bật sự khác biệt về trung vị và dải phân bố giữa hai nhóm.


In [ ]:
# 2.1 Chỉ số phát triển thành phố (city_development_index)
plot_numerical_feature(df, col='city_development_index', save_path='../reports/figures/group2_num_city_development_index.png', show=True)


In [ ]:
# 2.2 Số giờ đào tạo (training_hours)
plot_numerical_feature(df, col='training_hours', save_path='../reports/figures/group2_num_training_hours.png', show=True)


---
## NHÓM 3: BIẾN PHÂN LOẠI DANH NGHĨA ÍT GIÁ TRỊ (LOW-CARDINALITY NOMINAL)
**Biểu đồ sử dụng:** Phương án A (Đồng nhất & Chuyên sâu)
- **Subplot 1:** Bar Chart phân phối số lượng mẫu & % cơ cấu trong danh mục (bao gồm cả nhóm `(Missing)` nếu có).
- **Subplot 2:** Bar Chart tỷ lệ Target = 1 (%) của từng danh mục kèm đường nét đứt tham chiếu tỷ lệ trung bình toàn tập.


In [ ]:
# 3.1 Giới tính (gender)
plot_nominal_feature(df, col='gender', save_path='../reports/figures/group3_nominal_gender.png', show=True)


In [ ]:
# 3.2 Loại hình công ty (company_type)
plot_nominal_feature(df, col='company_type', save_path='../reports/figures/group3_nominal_company_type.png', show=True)


In [ ]:
# 3.3 Chuyên ngành đào tạo (major_discipline)
plot_nominal_feature(df, col='major_discipline', save_path='../reports/figures/group3_nominal_major_discipline.png', show=True)


In [ ]:
# 3.4 Kinh nghiệm liên quan (relevent_experience)
plot_nominal_feature(df, col='relevent_experience', save_path='../reports/figures/group3_nominal_relevent_experience.png', show=True)


In [ ]:
# 3.5 Đăng ký khóa học đại học (enrolled_university)
plot_nominal_feature(df, col='enrolled_university', save_path='../reports/figures/group3_nominal_enrolled_university.png', show=True)


---
## NHÓM 4: BIẾN PHÂN LOẠI THỨ BẬC (ORDINAL CATEGORICAL)
**Biểu đồ sử dụng:** 
- **Subplot 1:** Ordered Bar Chart (Sắp xếp đúng thứ tự cấp bậc tự nhiên).
- **Subplot 2:** Tỷ lệ Target = 1 (%) theo cấp bậc.
  - *Đặc biệt đối với biến `experience` (23 mốc giá trị):* Sử dụng **Line Chart có marker** thay cho Bar Chart để tránh chen chúc nhãn, đồng thời trực quan hóa cực kỳ trực tiếp xu hướng dốc xuống: "Số năm kinh nghiệm càng cao thì tỷ lệ muốn đổi việc càng giảm mạnh" (từ 45.4% ở `<1` năm xuống còn 15.3% ở `>20` năm).


In [ ]:
# 4.1 Trình độ học vấn (education_level)
edu_order = ['Primary School', 'High School', 'Graduate', 'Masters', 'Phd']
plot_ordinal_feature(df, col='education_level', order=edu_order, save_path='../reports/figures/group4_ordinal_education_level.png', show=True)


In [ ]:
# 4.2 Quy mô công ty (company_size)
size_order = ['<10', '10/49', '50-99', '100-500', '500-999', '1000-4999', '5000-9999', '10000+']
plot_ordinal_feature(df, col='company_size', order=size_order, save_path='../reports/figures/group4_ordinal_company_size.png', show=True)


In [ ]:
# 4.3 Số năm kinh nghiệm (experience) - Line Chart xu hướng dốc xuống
exp_order = ['<1'] + [str(i) for i in range(1, 21)] + ['>20']
plot_ordinal_feature(df, col='experience', order=exp_order, save_path='../reports/figures/group4_ordinal_experience.png', show=True)


In [ ]:
# 4.4 Khoảng cách công việc trước đây (last_new_job)
job_order = ['never', '1', '2', '3', '4', '>4']
plot_ordinal_feature(df, col='last_new_job', order=job_order, save_path='../reports/figures/group4_ordinal_last_new_job.png', show=True)


---
## NHÓM 5: BIẾN DANH NGHĨA NHIỀU GIÁ TRỊ (HIGH-CARDINALITY NOMINAL)
**Biểu đồ sử dụng:** Horizontal Bar Chart cho Top 15 đô thị tập trung đông ứng viên nhất kèm tỷ lệ Target=1 tương ứng.


In [ ]:
# 5.1 Top 15 Thành phố (city)
plot_high_cardinality_feature(df, col='city', top_n=15, save_path='../reports/figures/group5_high_cardinality_city_top15.png', show=True)


### 5.2 Kiểm chứng quan hệ 1 - 1 giữa `city` và `city_development_index` (CDI)
**Mục tiêu:** Kiểm tra xem mỗi mã `city` có ứng với duy nhất một giá trị `city_development_index` hay không, từ đó đưa ra quyết định kỹ thuật về việc giữ lại hay loại bỏ biến `city` trong Pipeline.


In [ ]:
check_city_cdi_relationship(df)


> **Nhận định & Quyết định kỹ thuật (Feature Selection):**
> 1. **Quan hệ 1 - 1 hoàn hảo:** 100% các thành phố (123/123) đều có duy nhất 1 giá trị `city_development_index`. Điều này chứng minh toàn bộ thông tin về vị trí địa lý / tiềm lực kinh tế của thành phố đã được số hóa đầy đủ vào chỉ số CDI.
> 2. **Tránh bùng nổ số chiều (Curse of Dimensionality):** Biến `city` có tới 123 danh mục. Nếu giữ lại và áp dụng One-Hot Encoding, mô hình sẽ bị thêm 123 cột rất thưa (sparse features), làm tăng đột biến chi phí tính toán và nguy cơ overfitting.
> 3. **Khuyến nghị cho Pipeline:** Loại bỏ biến `city` khỏi tập đặc trưng đầu vào, chỉ giữ lại `city_development_index` làm đại diện số liên tục.


---
## NHÓM 6: KHẢO SÁT DỮ LIỆU KHUYẾT & TƯƠNG QUAN
**Biểu đồ sử dụng:**
1. **Missing Values Bar Chart:** Tỷ lệ khuyết (%) và số dòng khuyết ở từng đặc trưng.
2. **MNAR Analysis:** So sánh tỷ lệ Target = 1 giữa nhóm "Có dữ liệu" và nhóm "Bị khuyết" để chứng minh cơ chế Missing Not At Random.
3. **Spearman Rank Correlation Heatmap (7x7):** Đo lường tương quan thứ hạng giữa biến số, biến thứ bậc và Target.


In [ ]:
# 6.1 Bảng thống kê khuyết
get_missing_stats(df)


In [ ]:
# 6.2 Biểu đồ tỷ lệ khuyết ở từng cột
plot_missing_values(df, save_path='../reports/figures/group6_missing_values_bar.png', show=True)


In [ ]:
# 6.3 Kiểm chứng cơ chế khuyết MNAR (Có dữ liệu vs Bị khuyết)
plot_missingness_vs_target(df, target_col='target', save_path='../reports/figures/group6_missingness_vs_target.png', show=True)


> **Phát hiện quan trọng về Cơ chế Khuyết (Missing Mechanism):**
> - Ở hai đặc trưng `company_size` và `company_type`, ứng viên **bị khuyết thông tin có tỷ lệ muốn đổi việc lên tới 40.6% và 38.3%**, cao gần gấp đôi so với nhóm có dữ liệu đầy đủ (~15% - 21%).
> - Đây là bằng chứng rõ ràng của cơ chế **MNAR (Missing Not At Random)**: Ứng viên không điền thông tin công ty vì họ là freelancer, sinh viên mới ra trường hoặc đang thất nghiệp. Việc không có công ty chính là động lực mạnh mẽ thúc đẩy họ tìm kiếm việc làm mới.
> - **Hệ quả kỹ thuật:** Tuyệt đối **không được drop** các dòng này và **không được điền bằng Mode** (vì sẽ làm méo mó bản chất của người đang có việc). Cần điền bằng nhãn riêng như `'Unknown'` và bổ sung `MissingIndicator`.


In [ ]:
# 6.4 Ma trận tương quan Spearman Rank Correlation (7x7)
plot_correlation_matrix(df, method='spearman', save_path='../reports/figures/group6_correlation_matrix.png', show=True)


> **Phân tích Ma trận Tương quan Spearman (7x7):**
> - **`city_development_index`** có tương quan âm mạnh nhất với `target` (r = **-0.341**): Ứng viên sống tại các đô thị phát triển ít có xu hướng đổi việc hơn đáng kể.
> - **`experience`** có tương quan âm với `target` (r = **-0.176**): Càng tích lũy nhiều năm kinh nghiệm, xu hướng đổi việc càng giảm dần.
> - **`company_size`** có tương quan âm với `target` (r = **-0.133**): Làm việc tại các công ty quy mô lớn đem lại tính ổn định cao hơn, giảm tỷ lệ tìm việc mới.


---
## NHÓM 7: KHẢO SÁT KHÔNG GIAN ĐẶC TRƯNG CHO KỸ THUẬT PCA (BÁM SÁT ĐỀ BÀI PDF)
**Mục tiêu (Mục 4.1 trang 3 đề bài PDF):**
- Đề bài yêu cầu áp dụng kỹ thuật PCA để giảm chiều dữ liệu và so sánh hiệu năng với mô hình huấn luyện trên toàn bộ đặc trưng gốc.
- Trước khi đưa vào huấn luyện mô hình, ta thực hiện phân tích thành phần chính trên không gian đặc trưng sau khi mã hóa One-Hot (đã loại bỏ `enrollee_id`, `city`, `target`), sau đó vẽ **Scree Plot (Cumulative Explained Variance Ratio)** để xác định số lượng Principal Components cần thiết bảo toàn **90%** và **95%** phương sai.


In [ ]:
# 7.1 Vẽ Scree Plot và phân tích số lượng thành phần chính PCA
plot_pca_scree_plot(df, save_path='../reports/figures/group7_pca_scree_plot.png', show=True)


> **Nhận định & Khuyến nghị kỹ thuật cho bước Giảm Chiều PCA:**
> - Không gian đặc trưng sau One-Hot Encoding (loại bỏ `enrollee_id`, `city`, `target`) có tổng cộng **62 chiều**.
> - **Ngưỡng 90% phương sai:** Cần chọn $k = 47$ components.
> - **Ngưỡng 95% phương sai:** Cần chọn $k = 51$ components.
> - **Ứng dụng cho Pipeline:** Trong giai đoạn huấn luyện mô hình học máy (Task 2 & Task 3 theo PDF), ta thiết lập cấu hình `PCA(n_components=0.90)` (tương ứng 47 components) để tiến hành so sánh đối chuẩn (benchmark) với mô hình Baseline không giảm chiều.


---
## TỔNG KẾT: CHIẾN LƯỢC TIỀN XỬ LÝ DỮ LIỆU & XỬ LÝ DỮ LIỆU KHUYẾT (IMPUTATION STRATEGY)

Dựa trên toàn bộ kết quả phân tích thống kê và kiểm chứng cơ chế khuyết ở trên, nhóm đề xuất bảng chiến lược tiền xử lý chuẩn mực cho Pipeline học máy như sau:

| Tên đặc trưng | Tỷ lệ khuyết (%) | Cơ chế khuyết dự đoán | Chiến lược xử lý đề xuất cho Pipeline | Căn cứ khoa học từ kết quả EDA |
| :--- | :--- | :--- | :--- | :--- |
| **`company_size`** | 30.82% | **MNAR** (Missing Not At Random) | Điền bằng nhóm riêng `'Unknown'` + Thêm `MissingIndicator` | Tỷ lệ đổi việc nhóm khuyết lên tới **40.6%** (gấp 2-3 lần nhóm có quy mô công ty). Nếu điền Mode sẽ làm sai lệch phân phối nghiêm trọng. |
| **`company_type`** | 32.05% | **MNAR** | Điền bằng nhóm riêng `'Unknown'` + Thêm `MissingIndicator` | Tỷ lệ đổi việc nhóm khuyết là **38.3%** so với 17-21% ở các loại hình công ty đã biết. |
| **`gender`** | 23.53% | **MNAR / MAR** | Điền bằng nhóm riêng `'Unknown'` hoặc `'Other'` | Nhóm khuyết có tỷ lệ đổi việc **30.5%** (cao hơn nam giới 22.7%). Ứng viên có thể e ngại tiết lộ thông tin giới tính khi nộp hồ sơ. |
| **`major_discipline`** | 14.68% | **MAR** (Phụ thuộc `education_level`) | Điền bằng `'No Major'` hoặc `'Non-STEM'` | Đa số ứng viên khuyết chuyên ngành là do học vấn ở bậc High School hoặc Primary School (vốn không có chuyên ngành đại học). |
| **`education_level`** | 2.40% | **MAR / MCAR** | Điền bằng giá trị phổ biến nhất (Mode: `'Graduate'`) hoặc KNN Imputer | Tỷ lệ khuyết rất nhỏ (2.4%), không tạo ra sự chênh lệch đột biến về tỷ lệ Target. |
| **`last_new_job`** | 2.21% | **MAR** | Điền bằng Mode (`'1'`) hoặc gộp vào nhóm `'never'` | Tỷ lệ khuyết thấp (2.21%), đa số ứng viên rơi vào mốc đổi việc 1 năm trước. |
| **`enrolled_university`** | 2.01% | **MAR** | Điền bằng Mode (`'no_enrollment'`) | Tỷ lệ khuyết chỉ 2.01%, nhóm `no_enrollment` chiếm đa số áp đảo (>72%). |
| **`experience`** | 0.34% | **MCAR** (Missing Completely At Random) | Điền bằng Median (khoảng 9 năm) hoặc Mode (`'>20'`) | Tỷ lệ khuyết không đáng kể (0.34%), phân bố ngẫu nhiên hoàn toàn. |
| **`city`** | 0.00% | *Không khuyết* | **Loại bỏ khỏi tập đặc trưng (Drop column)** | Đã chứng minh quan hệ 1-1 với `city_development_index`. Loại bỏ để tránh tạo 123 cột thưa gây hiện tượng nguyền rủa số chiều. |
| **`enrollee_id`** | 0.00% | *Không khuyết* | **Loại bỏ khỏi tập đặc trưng (Drop column)** | Cột định danh ngẫu nhiên, không mang giá trị thông tin học máy. |
